# 1 — US Accidents: Model-Free Preprocessing (Weekly Count Foundation)

## Objective

Replace the old event-level `Severity` classification data flow with a compact, model-independent **weekly accident-count foundation**. The only event fields needed at this stage are `Start_Time` and `State`:

- Every source row with a valid `Start_Time` is counted exactly once.
- No severity filtering, latitude/longitude filtering, top-state selection, or event sampling is applied.
- Each report is assigned to a Monday-start local-calendar week using the source `Start_Time` clock.
- State is retained for the graph-based DCRNN stream. Missing state values are retained in an explicit `__UNKNOWN__` node rather than silently discarded.
- The event-level source is scanned with Polars lazy execution; only the compact `week_start × state` aggregate is written.

`week_id` is added in Notebook 2 after the full ordered weekly index is constructed. `week_start` remains a timestamp for interpretation, calendar features and plots; `week_id` is the sequential grouping key requested for batching.

## Important time convention

The source dataset documents `Start_Time` as local time. We therefore group by the local calendar week present in each record instead of incorrectly treating all wall-clock values as UTC. The national sum combines each report's local Monday-start week. This convention is explicit and used consistently by both streams.

## Input

Attach the Kaggle US Accidents dataset (`sobhanmoosavi/us-accidents`) as an input. This notebook expects `US_Accidents_March23.csv`.

## 0. Dependencies and paths

Kaggle normally provides Polars. If a dependency is missing, uncomment the installation line. This notebook deliberately does not install or upgrade PyTorch, CUDA, or Scikit-learn.

In [ ]:
# Uncomment only if the Kaggle environment is missing a required package:
# %pip install -q "polars>=1.20,<2" pyarrow

from pathlib import Path
import os, json, time
import polars as pl

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
REPORT_DIR = PROJECT_ROOT / 'reports'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = None  # Optionally set to the exact CSV path.
KAGGLE_DATASET = 'sobhanmoosavi/us-accidents'

## 1. Locate the source CSV

The source path is resolved automatically from `/kaggle/input`. If it is not attached, the notebook reports the required input rather than silently downloading a different copy.

In [ ]:
def locate_csv():
    if DATA_FILE is not None:
        p = Path(DATA_FILE)
        if not p.exists():
            raise FileNotFoundError(f'DATA_FILE does not exist: {p}')
        return p
    roots = [Path('/kaggle/input'), Path('.')]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob('US_Accidents_March23.csv'))
    if not matches:
        raise FileNotFoundError(
            'US_Accidents_March23.csv was not found. In Kaggle, use Add Input and attach '
            'sobhanmoosavi/us-accidents, or set DATA_FILE to the CSV path.'
        )
    # Prefer the largest exact-name match; duplicate copies of the same dataset are common.
    return max(matches, key=lambda p: p.stat().st_size)

CSV_PATH = locate_csv()
print('CSV:', CSV_PATH)
print('Size (GiB):', round(CSV_PATH.stat().st_size / 1024**3, 3))
print('Polars:', pl.__version__)

## 2. Inspect required columns and lazily scan the source

Only `Start_Time` and `State` are required. This is intentional: the forecasting target is the number of reports, not severity, weather, or a severity-derived proxy.

In [ ]:
lf = pl.scan_csv(
    str(CSV_PATH),
    infer_schema_length=10000,
    ignore_errors=True,
    schema_overrides={'Start_Time': pl.String, 'State': pl.String},
    low_memory=True,
)
source_columns = lf.collect_schema().names()
required = {'Start_Time', 'State'}
missing = sorted(required - set(source_columns))
if missing:
    raise ValueError(f'Required columns missing from source CSV: {missing}')
print('Source columns:', len(source_columns))

## 3. Parse timestamps, preserve unknown state values, and audit rejected timestamps

A row with a missing state is retained as `__UNKNOWN__`, so it still contributes to the national count and remains represented in the graph data. A row with an unparseable/missing `Start_Time` cannot be assigned to a week; its count is explicitly reported in the audit.

In [ ]:
parsed = lf.select([
    pl.col('Start_Time').cast(pl.String).str.to_datetime(strict=False, exact=False, time_unit='us').alias('timestamp'),
    pl.col('State').cast(pl.String).str.strip_chars().str.to_uppercase().alias('state'),
])

audit = parsed.select([
    pl.len().alias('source_rows'),
    pl.col('timestamp').is_null().sum().alias('invalid_timestamp_rows'),
    pl.col('state').is_null().sum().alias('null_state_rows_before_fill'),
    pl.col('timestamp').min().alias('first_valid_timestamp'),
    pl.col('timestamp').max().alias('last_valid_timestamp'),
]).collect(engine='streaming')
source_rows = int(audit['source_rows'][0])
invalid_timestamp_rows = int(audit['invalid_timestamp_rows'][0])
null_state_rows = int(audit['null_state_rows_before_fill'][0])
print(audit)
if source_rows == 0:
    raise ValueError('The CSV scan returned zero rows.')
if invalid_timestamp_rows:
    raise ValueError(
        f'{invalid_timestamp_rows:,} source rows have missing/unparseable Start_Time. '
        'They cannot be assigned to a weekly target. Inspect the source values and timestamp parser; '
        'do not continue until every accident row has a valid weekly assignment.'
    )

## 4. Aggregate every valid report into a weekly state count

Polars `truncate('1w')` uses Monday-aligned weekly windows. The grouped result is sparse: a state-week pair with no row is filled with zero in Notebook 2 when the complete panel is constructed.

In [ ]:
weekly_sparse_lf = (
    parsed
    .filter(pl.col('timestamp').is_not_null())
    .with_columns([
        pl.col('state').fill_null('__UNKNOWN__').replace('', '__UNKNOWN__').alias('state'),
        pl.col('timestamp').dt.truncate('1w').alias('week_start'),
    ])
    .group_by(['week_start', 'state'])
    .agg(pl.len().cast(pl.Int64).alias('accident_count'))
    .sort(['week_start', 'state'])
)

weekly_sparse_path = ARTIFACT_DIR / 'weekly_state_counts_sparse.parquet'
weekly_sparse_path.unlink(missing_ok=True)
weekly_sparse_lf.sink_parquet(weekly_sparse_path, compression='zstd')
print('Saved:', weekly_sparse_path)

weekly_sparse = pl.read_parquet(weekly_sparse_path)
valid_timestamp_rows = int(weekly_sparse['accident_count'].sum())
if valid_timestamp_rows != source_rows - invalid_timestamp_rows:
    raise AssertionError(
        f'Row conservation failed: aggregated={valid_timestamp_rows:,}, '
        f'expected={source_rows-invalid_timestamp_rows:,}'
    )

## 5. Save source and aggregation metadata

In [ ]:
state_values = sorted(weekly_sparse['state'].unique().to_list())
week_values = weekly_sparse['week_start'].unique().sort().to_list()
metadata = {
    'source_file': CSV_PATH.name,
    'source_path_at_run': str(CSV_PATH),
    'dataset': KAGGLE_DATASET,
    'source_rows': source_rows,
    'invalid_timestamp_rows': invalid_timestamp_rows,
    'valid_timestamp_rows_aggregated': valid_timestamp_rows,
    'null_state_rows_before_fill': null_state_rows,
    'first_valid_timestamp': str(audit['first_valid_timestamp'][0]),
    'last_valid_timestamp': str(audit['last_valid_timestamp'][0]),
    'state_values': state_values,
    'n_states_including_unknown_bucket': len(state_values),
    'first_week_start': str(min(week_values)) if week_values else None,
    'last_week_start': str(max(week_values)) if week_values else None,
    'week_definition': 'Monday-start local-calendar week from source Start_Time',
    'row_policy': 'All rows with valid Start_Time retained; no state/severity/geographic filtering.',
}
(ARTIFACT_DIR / 'source_aggregation_metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps({k: metadata[k] for k in [
    'source_rows', 'invalid_timestamp_rows', 'valid_timestamp_rows_aggregated',
    'n_states_including_unknown_bucket', 'first_week_start', 'last_week_start'
]}, indent=2))
print('State codes:', state_values)

## 6. Completion checks and outputs

Expected outputs:

- `artifacts/weekly_state_counts_sparse.parquet`
- `artifacts/source_aggregation_metadata.json`

**Important:** the Parquet file contains counts only for observed state-week pairs. Notebook 2 creates the complete regular week index, assigns sequential `week_id` values starting at 1, fills missing state-week combinations with zero, performs EDA, and makes the chronological train/test split at its end.